# Score Matching Toy Objective

Chapter 17: Score Function and Energy-Based Models · *Mathematics for AI and Machine Learning* by Fuheng Wu

Compare true vs model score and the squared mismatch objective.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-17/score-matching/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

torch.manual_seed(0)

# The demo's setup: data ~ N(0, 0.9^2); Gaussian model N(mu, sig^2) with defaults mu = 1.2, sig = 1.5
data_mu, data_sig = 0.0, 0.9
mu0, sig0 = 1.2, 1.5
x = data_mu + data_sig * torch.randn(200000, dtype=torch.float64)


def model_score(x, mu, sig):
    return -(x - mu) / sig ** 2


# ── Fisher divergence E_p[(s_model - s_data)^2] needs the data's true score ─
fisher = ((model_score(x, mu0, sig0) - model_score(x, data_mu, data_sig)) ** 2).mean()
print(f"Fisher divergence at mu = {mu0}, sig = {sig0}: {fisher.item():.4f}")

# ── Hyvarinen's trick: the same objective (up to a constant) from samples only
#    J = E_p[ s(x)^2 / 2 + s'(x) ], with s'(x) = -1 / sig^2 for a Gaussian model
def implicit_sm(mu, sig):
    s = model_score(x, mu, sig)
    return (0.5 * s ** 2 - 1 / sig ** 2).mean()


const = 0.5 * (model_score(x, data_mu, data_sig) ** 2).mean()
print(f"implicit objective J = {implicit_sm(mu0, sig0).item():.4f};  J + E[s_data^2]/2 = "
      f"{(implicit_sm(mu0, sig0) + const).item():.4f}  (~ Fisher / 2 = {fisher.item() / 2:.4f}; equal in expectation)")

# ── Minimize J by gradient descent: recovers the data's mean and std ────────
mu = torch.tensor(mu0, dtype=torch.float64, requires_grad=True)
log_sig = torch.log(torch.tensor(sig0, dtype=torch.float64)).requires_grad_(True)
opt = torch.optim.Adam([mu, log_sig], lr=0.05)
for _ in range(600):
    opt.zero_grad()
    implicit_sm(mu, log_sig.exp()).backward()
    opt.step()
print(f"\nscore matching fit: mu = {mu.item():+.4f} (data {data_mu}), sig = {log_sig.exp().item():.4f} (data {data_sig})")
print(f"sample mean {x.mean().item():+.4f}, sample std {x.std().item():.4f}")

Fisher divergence at mu = 1.2, sig = 1.5: 0.7895
implicit objective J = -0.2219;  J + E[s_data^2]/2 = 0.3958  (~ Fisher / 2 = 0.3948; equal in expectation)



score matching fit: mu = -0.0011 (data 0.0), sig = 0.9003 (data 0.9)
sample mean -0.0011, sample std 0.9003
